# SMS spam detection: an NLP baseline

**Question:** Can message text be triaged to reduce spam exposure without blocking legitimate messages?

Source: [UCI SMS Spam Collection](https://archive.ics.uci.edu/dataset/228/sms+spam+collection). False positives are especially costly, so precision and recall are both reported.

In [ ]:
from pathlib import Path
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, average_precision_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline

df=pd.read_csv(Path('../datasets/portfolio-expansion/sms-spam/SMSSpamCollection'),sep='\t',header=None,names=['label','message'])
X_train,X_test,y_train,y_test=train_test_split(df.message,df.label.eq('spam'),test_size=.2,stratify=df.label,random_state=42)
# TF-IDF turns text into weighted word/phrase features without hand-written rules.
model=Pipeline([('tfidf',TfidfVectorizer(lowercase=True,stop_words='english',ngram_range=(1,2),min_df=2)),('classifier',LogisticRegression(max_iter=2000,class_weight='balanced'))]).fit(X_train,y_train)
probability=model.predict_proba(X_test)[:,1]; prediction=(probability>=.5).astype(int)
print('PR-AUC:',round(average_precision_score(y_test,probability),3)); print(classification_report(y_test,prediction,target_names=['ham','spam']))

In [ ]:
# Inspect the strongest linear signals to make the model behaviour reviewable.
terms=model.named_steps['tfidf'].get_feature_names_out(); weights=model.named_steps['classifier'].coef_[0]
signals=pd.DataFrame({'term':terms,'weight':weights}).sort_values('weight',ascending=False)
display(signals.head(20)); display(signals.tail(20))
# Do not log or expose message contents in a production monitoring system without privacy controls.

## Portfolio takeaway

This is a transparent, strong baseline. Next, tune the decision threshold for the moderation workflow and evaluate drift as spam campaigns evolve.